# DriftEnv — GRPO Training Run

**Project:** DriftEnv — RL environment testing AI agent robustness under ambiguity and context shift  
**Live Space:** https://huggingface.co/spaces/harims95/driftenv  
**Repo:** https://github.com/harims95/driftenv  
**Training date:** <!-- FILL IN: e.g. April 26, 2026 -->  
**GPU:** A10G (switch from T4 after dry run confirms no errors)  
**Model:** Qwen2.5-1.5B-Instruct via Unsloth 4-bit  

---

**What this notebook does:**
1. Loads Qwen2.5-1.5B-Instruct with Unsloth (4-bit, LoRA)
2. Trains with GRPO against the live DriftEnv Space (20 training scenarios)
3. Evaluates untrained vs trained on 5 holdout scenarios
4. Saves reward curves and before/after bar chart to `assets/`

**Run order:** cells top-to-bottom. Do NOT skip Cell 2 (installs).

In [ ]:
# Run once per Colab session — kernel reset wipes installed packages
!pip install -q unsloth trl peft accelerate datasets
!pip install -q httpx pydantic huggingface_hub

# Verify key installs
import unsloth, trl
print(f"unsloth: {unsloth.__version__}  |  trl: {trl.__version__}")

In [ ]:
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from huggingface_hub import login
from datasets import Dataset
import httpx, json, os

# Authenticate — paste your HF token when prompted
# Token needs: read + Inference API access
login()

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
MODEL_NAME    = "unsloth/Qwen2.5-1.5B-Instruct"  # Unsloth pre-optimized
MAX_SEQ_LEN   = 2048
LORA_RANK     = 16
DRIFTENV_URL  = "https://harims95-driftenv.hf.space"
HOLDOUT_ONLY  = False  # False during training; flip to True for eval runs

# Pre-warm the Space (first cold-start request can take ~30s)
import httpx as _hx
try:
    r = _hx.get(DRIFTENV_URL, timeout=60)
    print("Space status:", r.json().get("status"))
except Exception as e:
    print("Space warm-up failed — check URL:", e)

## Load Model with Unsloth

Uses 4-bit quantization to fit in 24 GB VRAM (A10G).  
**Critical: do NOT replace with plain `transformers.AutoModelForCausalLM`.**  
Unsloth's `FastLanguageModel` provides ~2x faster rollout generation —  
rollouts dominate RL runtime, so this is not optional.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,  # auto-detect
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth optimised checkpointing
)

# Confirm trainable params — should be ~10-15M for 1.5B + rank-16 LoRA
model.print_trainable_parameters()

## DriftEnv Client Wrapper

Talks to the live HF Space over HTTP.  
If the Space is cold, the first request takes ~30s — Cell 4 pre-warms it.  
All calls are synchronous (httpx blocking) — fine for sequential rollouts.

In [ ]:
def reset_env(task: str = "medium", holdout_only: bool = HOLDOUT_ONLY) -> dict:
    r = httpx.post(
        f"{DRIFTENV_URL}/reset",
        json={"task": task, "holdout_only": holdout_only},
        timeout=60,
    )
    r.raise_for_status()
    return r.json()

def step_env(action_text: str) -> dict:
    r = httpx.post(
        f"{DRIFTENV_URL}/step",
        json={"action": {"response": action_text}},
        timeout=60,
    )
    r.raise_for_status()
    return r.json()

# Smoke test
obs = reset_env(task="easy", holdout_only=False)
print("reset ok — instruction:", obs["observation"]["instruction"])

## Rollout Function (GRPO Reward Function)

This is the bridge between GRPO and DriftEnv.  
For each completion, it calls `/step`, gets the 4-component reward,  
and returns the weighted scalar to the trainer.

**Critical:** call `FastLanguageModel.for_inference(model)` before generating  
rollouts — this activates Unsloth's 2x inference kernel. Easy to forget.  
After training resumes, call `FastLanguageModel.for_training(model)`.

The `reward_log` global accumulates per-step component values for plotting.

In [ ]:
# Global log — each entry: {step, R_format, R_interpretation, R_pivot, R_no_stale, total}
reward_log = []

def driftenv_reward_fn(prompts: list, completions: list, **kwargs) -> list:
    """
    GRPO reward function. Called once per batch of rollouts.

    prompts:     list of prompt strings (one per rollout)
    completions: list of completion strings (one per rollout)
    returns:     list of scalar reward floats
    """
    # TODO: fill in tomorrow
    # Skeleton:
    #
    # FastLanguageModel.for_inference(model)   # <-- CRITICAL, 2x speedup
    # rewards = []
    # for prompt, completion in zip(prompts, completions):
    #     result = step_env(completion)
    #     components = result["info"].get("rewards", {})
    #     total = result["reward"]
    #     reward_log.append({
    #         "step": len(reward_log),
    #         **components,
    #         "total": total,
    #     })
    #     rewards.append(total)
    # FastLanguageModel.for_training(model)    # switch back for grad updates
    # return rewards
    raise NotImplementedError("Fill in reward function before running")

## GRPO Config

150 training steps, batch 4, 4 generations per prompt.  
Total rollouts per step = `per_device_train_batch_size × num_generations` = 16.  

**For T4 dry run:** set `max_steps=3` to verify pipeline without burning credits.  
**For A10G real run:** use `max_steps=150` as configured below.

In [ ]:
config = GRPOConfig(
    output_dir="driftenv_grpo_run",
    max_steps=150,                      # set to 3 for T4 dry run
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_generations=4,                  # GRPO group size — 4 completions compared per prompt
    learning_rate=1e-5,
    logging_steps=5,
    save_steps=50,
    max_prompt_length=512,
    max_completion_length=256,
    temperature=0.9,
    report_to="none",                   # manual plotting below
)

print("Config ready. max_steps:", config.max_steps)
print("Rollouts per step:", config.per_device_train_batch_size * config.num_generations)

## Build Prompt Dataset

Fetch the 20 training scenarios from the live Space (holdout excluded).  
Format each as a prompt string matching the system prompt in `inference.py`.  
Wrap in a HuggingFace `Dataset` object for the trainer.

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert AI/ML engineer working on real AI development tasks.\n"
    "You will receive task instructions that may be vague or incomplete.\n"
    "Your job at each step:\n"
    "  1. Carefully interpret what the instruction actually requires technically.\n"
    "  2. If requirements change mid-task, immediately acknowledge and adapt.\n"
    "  3. Always give a specific, technically detailed response.\n"
    "Be concise but precise. State your interpretation clearly."
)

def build_prompt_dataset(n_samples: int = 80) -> Dataset:
    """
    Build training prompts by sampling from the 20 training scenarios.
    Each sample = reset env (holdout_only=False) + format first observation as prompt.
    n_samples should be >= max_steps to avoid dataset exhaustion.
    """
    # TODO: fill in tomorrow
    # Skeleton:
    #
    # prompts = []
    # for _ in range(n_samples):
    #     obs = reset_env(task="medium", holdout_only=False)
    #     instruction = obs["observation"]["instruction"]
    #     prompt = (
    #         f"{SYSTEM_PROMPT}\n\n"
    #         f'Task: "{instruction}"\n'
    #         "This instruction is vague. State your full technical interpretation."
    #     )
    #     prompts.append({"prompt": prompt})
    # return Dataset.from_list(prompts)
    raise NotImplementedError("Fill in dataset builder before running")

# prompt_dataset = build_prompt_dataset(n_samples=80)
# print(f"Dataset ready: {len(prompt_dataset)} prompts")
# print("Sample prompt:\n", prompt_dataset[0]["prompt"][:300])

## Train

Initialise `GRPOTrainer` and call `.train()`.  
Expected time on A10G: ~2–3 hours for 150 steps.  
Watch the first 5 logging steps — reward should be non-zero and non-NaN.  
If reward is flat for 20+ steps, something is wrong — stop and debug on T4 first.

In [ ]:
# Uncomment after filling in Cell 10 (reward fn) and Cell 14 (dataset)

# prompt_dataset = build_prompt_dataset(n_samples=80)

# trainer = GRPOTrainer(
#     model=model,
#     reward_funcs=[driftenv_reward_fn],
#     args=config,
#     train_dataset=prompt_dataset,
#     tokenizer=tokenizer,
# )

# trainer.train()

## Save Trained LoRA Adapter

Save the adapter only — do NOT merge into the base model weights.  
Unsloth has a specific merge function (`model.save_pretrained_merged`) if you  
need a merged checkpoint later, but for the demo the adapter is sufficient.  
Push to HF Hub under your namespace so it's accessible for the eval cell.

In [ ]:
ADAPTER_DIR  = "driftenv_grpo_adapter"
HF_REPO_ID   = "harims95/driftenv-qwen1.5b-grpo"  # will be created if it doesn't exist

# model.save_pretrained(ADAPTER_DIR)
# tokenizer.save_pretrained(ADAPTER_DIR)
# print(f"Adapter saved locally to {ADAPTER_DIR}/")

# model.push_to_hub(HF_REPO_ID, token=True)
# tokenizer.push_to_hub(HF_REPO_ID, token=True)
# print(f"Adapter pushed to https://huggingface.co/{HF_REPO_ID}")

## Evaluation — Untrained vs Trained on Holdout

Run both model variants against the 5 holdout scenarios (IDs 1,3,7,14,20).  
Record all 4 reward components per step.  
Save to JSON — these become the before/after numbers in the README.

**Three numbers to report:**
1. Untrained Qwen 1.5B on holdout  
2. Trained Qwen 1.5B on holdout ← headline metric  
3. Qwen 72B reference on holdout: **~0.378** (already have this)

In [ ]:
def eval_on_holdout(label: str, generate_fn, n_episodes: int = 5) -> dict:
    """
    Run generate_fn against the holdout set and collect reward components.

    generate_fn: callable(prompt: str) -> str  (wraps model.generate or a reference model)
    Returns dict with per-episode and aggregate results.
    """
    # TODO: fill in tomorrow
    # Skeleton:
    #
    # results = []
    # for _ in range(n_episodes):
    #     obs = reset_env(task="medium", holdout_only=True)
    #     episode = {"rewards": [], "components": []}
    #     for step_num in range(1, 4):
    #         if obs["observation"]["done"]:
    #             break
    #         prompt = format_prompt(obs["observation"], step_num)  # reuse inference.py logic
    #         action = generate_fn(prompt)
    #         result = step_env(action)
    #         episode["rewards"].append(result["reward"])
    #         episode["components"].append(result["info"].get("rewards", {}))
    #         obs = result
    #     results.append(episode)
    # return {"label": label, "episodes": results,
    #         "mean_reward": sum(e["rewards"][-1] for e in results) / len(results)}
    raise NotImplementedError("Fill in eval function before running")

# FastLanguageModel.for_inference(model)  # CRITICAL before generating
# trained_results   = eval_on_holdout("trained_1.5B",   generate_fn=lambda p: ...)
# untrained_results = eval_on_holdout("untrained_1.5B", generate_fn=lambda p: ...)

# with open("../samples/eval_holdout_trained.json", "w") as f:
#     json.dump(trained_results, f, indent=2)
# with open("../samples/eval_holdout_untrained.json", "w") as f:
#     json.dump(untrained_results, f, indent=2)
# print("Untrained 1.5B holdout:", untrained_results["mean_reward"])
# print("Trained   1.5B holdout:", trained_results["mean_reward"])
# print("72B reference  holdout: 0.378")

## Plotting

Two figures saved to `assets/` in the repo root:

1. **`reward_curves.png`** — 4 reward components over training steps (from `reward_log`)
2. **`before_after.png`** — grouped bar chart: untrained 1.5B vs trained 1.5B vs 72B reference, per difficulty tier

Commit both PNGs to `multi-reward-v2` and embed in README.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ASSETS_DIR = "../assets"
os.makedirs(ASSETS_DIR, exist_ok=True)

def plot_reward_curves(log: list, save_path: str) -> None:
    """
    Plot the 4 reward component curves over training steps.
    log: list of dicts from reward_log (populated during training)
    """
    # TODO: fill in tomorrow after training completes
    # Skeleton:
    #
    # steps = [e["step"] for e in log]
    # fig, ax = plt.subplots(figsize=(10, 5))
    # for component, color in [("R_format", "grey"), ("R_interpretation", "blue"),
    #                           ("R_pivot", "green"), ("R_no_stale", "red")]:
    #     vals = [e.get(component, 0) for e in log]
    #     ax.plot(steps, vals, label=component, color=color, alpha=0.8)
    # ax.set_xlabel("Training step")
    # ax.set_ylabel("Reward component")
    # ax.set_title("DriftEnv — GRPO reward components over training")
    # ax.legend()
    # fig.tight_layout()
    # fig.savefig(save_path, dpi=150)
    # print(f"Saved: {save_path}")
    pass


def plot_before_after(untrained: dict, trained: dict, save_path: str) -> None:
    """
    Grouped bar chart: untrained 1.5B vs trained 1.5B vs 72B reference.
    One group per difficulty tier (easy / medium / hard).
    """
    # TODO: fill in tomorrow after eval_on_holdout runs
    # Skeleton:
    #
    # tiers = ["easy", "medium", "hard"]
    # ref_scores = {"easy": 0.307, "medium": 0.370, "hard": 0.611}  # 72B reference
    # x = np.arange(len(tiers))
    # width = 0.25
    # fig, ax = plt.subplots(figsize=(8, 5))
    # ax.bar(x - width, [untrained[t] for t in tiers], width, label="Untrained 1.5B", color="#d9534f")
    # ax.bar(x,         [trained[t]   for t in tiers], width, label="Trained 1.5B",   color="#5cb85c")
    # ax.bar(x + width, [ref_scores[t] for t in tiers], width, label="72B reference",  color="#5bc0de", alpha=0.7)
    # ax.set_xticks(x); ax.set_xticklabels(tiers)
    # ax.set_ylabel("Mean reward")
    # ax.set_title("DriftEnv — Before vs After GRPO training (holdout set)")
    # ax.legend()
    # fig.tight_layout()
    # fig.savefig(save_path, dpi=150)
    # print(f"Saved: {save_path}")
    pass


# plot_reward_curves(reward_log,   save_path=f"{ASSETS_DIR}/reward_curves.png")
# plot_before_after(untrained_results, trained_results,
#                   save_path=f"{ASSETS_DIR}/before_after.png")